# This notebook compute program and pathway signature score on the MIT excitatory neurons
We do it in python since the objects are too big to be loaded in R. 

In [ ]:
import os
import shutil
import scanpy as sc
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib
import copy
from matplotlib import pyplot as plt
from matplotlib import cm as cm
%matplotlib inline
from scipy import stats as sp_stats
import warnings
from datetime import datetime

warnings.filterwarnings("ignore")
sc.settings.n_jobs = 32
sc.set_figure_params(scanpy=True, dpi=100, dpi_save=500, frameon=False, vector_friendly=True, figsize=(10,10), format='png')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams["axes.grid"] = False


In [ ]:
from validation.validation_functions import get_shared_dir, get_result_dir, TOPICS_DATA_H5_PATH


DATA_DIR = os.path.join(get_shared_dir(), "data_Single-cell atlas reveals correlates of high cognitive function, dementia, and resilience to Alzheimers disease pathology")


CELL_TYPE_TO_GROUP = {
    "inhibitory": "neuronal/inhibitory",
    "cux2p": "neuronal/excitatory/cux2plus",
    "cux2m": "neuronal/excitatory/cux2minus",
    "excitatory": "neuronal/excitatory",
    "microglia": "glia/microglia",
    "astrocytes": "glia/astrocytes",
    "oligo": "glia/oligodendrocytes",
    "opc": "glia/opcs",
}


### Load the needed data files

In [ ]:
adata1 = sc.read_h5ad(os.path.join(DATA_DIR, "Excitatory_neurons_set1.h5ad"))
adata2 = sc.read_h5ad(os.path.join(DATA_DIR, "Excitatory_neurons_set2.h5ad"))
adata3 = sc.read_h5ad(os.path.join(DATA_DIR, "Excitatory_neurons_set3.h5ad"))

In [ ]:
combined_adata = sc.concat([adata1, adata2, adata3])

In [ ]:
# Save combined adata
combined_adata.write_h5ad(os.path.join(DATA_DIR, "Excitatory_neurons_combined.h5ad"))


In [ ]:
combined_adata = sc.read_h5ad(os.path.join(DATA_DIR, "Excitatory_neurons_combined.h5ad"))


In [ ]:
combined_adata.layers["old"] = combined_adata.X.copy()
sc.pp.normalize_total(combined_adata)
# Logarithmize the data
sc.pp.log1p(combined_adata)

In [ ]:
combined_adata.write_h5ad(os.path.join(DATA_DIR, "Excitatory_neurons_combined_normalized.h5ad"))


In [ ]:
combined_adata = sc.read(os.path.join(DATA_DIR, "Excitatory_neurons_combined_normalized.h5ad"))

In [ ]:
import pathlib

cell_type = "excitatory"
project = "mit"
root_plots_dir = "./Plots/mit"

plots_dir = f"{root_plots_dir}/{cell_type}"
try:
    os.mkdir(root_plots_dir)
except FileExistsError:
    pass
try:
    os.mkdir(plots_dir)
except FileExistsError:
    pass


In [ ]:

import h5py
import pandas as pd

if cell_type == "excitatory":
    with h5py.File(TOPICS_DATA_H5_PATH, "r") as h5file:
        des_cux2p = pd.DataFrame(h5file[CELL_TYPE_TO_GROUP["cux2p"] + '/de'][()])
    with h5py.File(TOPICS_DATA_H5_PATH, "r") as h5file:
        des_cux2m = pd.DataFrame(h5file[CELL_TYPE_TO_GROUP["cux2m"] + '/de'][()])
    des = pd.concat([des_cux2p, des_cux2m], ignore_index=True)
    if "cell.type" in des.columns:
        des["topic"] = des["cell.type"].astype(str) + "_" + des["topic"].astype(str)
else:
    with h5py.File(TOPICS_DATA_H5_PATH, "r") as h5file:
        des = pd.DataFrame(h5file[CELL_TYPE_TO_GROUP[cell_type] + '/de'][()])

# convert gene and topic to string
des["gene"] = des["gene"].astype(str)
des["topic"] = des["topic"].astype(str)
genes_by_topic = (
    des.groupby('topic')['gene']
    .apply(list)
    .to_dict()
)


In [ ]:
neuronal_topics_to_annotation = {
  "decreasing_early":("inh_k1","cux2p_k9", "cux2m_k4"),
  "decreasing_late":("inh_k7","cux2p_k10", "cux2m_k8"),
  "increasing_early":("inh_k10","cux2p_k11", "cux2m_k3"),
  "increasing_late":("inh_k8","cux2p_k4", "cux2m_k9")
}


In [ ]:
for union_name, topic_names in neuronal_topics_to_annotation.items():
# Apply convert_topic_name to each key in topic_names_raw
    print(topic_names)
    # Only keep topic nabmes that exist in genes_by_topic
    topic_names = [name for name in topic_names if name in genes_by_topic]
    if len(topic_names) > 0:
        # Gather all genes from selected topics, flatten, take unique
        union_genes = list(set(gene for t in topic_names for gene in genes_by_topic[t]))
        genes_by_topic[union_name] = union_genes


genes_by_topic = {k: genes_by_topic[k] for k in genes_by_topic if k in ['decreasing_early', 'decreasing_late', 'increasing_early', 'increasing_late']}

In [ ]:
# Create a list of genes per topic that don't appear in other topics
exclusive_genes_by_topic = {}
for topic, genes in genes_by_topic.items():
    other_genes = set(
        gene
        for other_topic, other_genes_list in genes_by_topic.items() 
        if other_topic != topic
        for gene in other_genes_list
    )
    exclusive = [gene for gene in genes if gene not in other_genes]
    exclusive_genes_by_topic[topic] = exclusive

In [ ]:
np.array([len(v) for _,v in genes_by_topic.items()]) - [len(v) for _,v in exclusive_genes_by_topic.items()]

In [ ]:
import validation
import h5py
from validation.validation_functions import compute_coherence_multiple

SHOULD_COMPUTE_COHERENCE = True
if cell_type in ("inhibitory", "excitatory") and SHOULD_COMPUTE_COHERENCE:
    n_cores = int(os.environ['SLURM_CPUS_ON_NODE'])

    if cell_type == "inhibitory":
        programs_to_check = [f"k{i}" for i in (1,7,8,10)]
        gene_sets_to_test = {k:v for k,v in exclusive_genes_by_topic.items() if k in programs_to_check}
    else:
        gene_sets_to_test = genes_by_topic.copy()
    expr_matrix = combined_adata.X.T

    # Compute coherence for all programs
    coherence_results = compute_coherence_multiple(expr_matrix, gene_sets_to_test, gene_names=combined_adata.var_names, n_permutations=100, n_cores=n_cores, temp_folder="/ems/elsc-labs/habib-n/roi.meir/tmp/joblib")
    coherence_results.to_pickle(os.path.join(plots_dir, "coherence_results.pickle"))
    
    # Convert strings to fixed-length bytes
    coherence_results['gene_set'] = coherence_results['gene_set'].astype(str)
    coherence_results['gene_set'] = np.array([str(x).encode('utf-8') for x in coherence_results['gene_set']], dtype='S')

    with h5py.File(TOPICS_DATA_H5_PATH, "a") as h5f:
        group_path = f"{CELL_TYPE_TO_GROUP[cell_type]}/{project}"
        h5f.create_dataset(f"{group_path}/coherence_results", data=coherence_results.drop(columns=['bg_means_cor'], inplace=False).to_records(index=False))
        bg_means_list = coherence_results['bg_means_cor'].tolist()
        # Stack into 2D array
        bg_means_2d = np.vstack([np.array(x) for x in bg_means_list])
        h5f.create_dataset(f"{group_path}/coherence_results_bg_means", data=bg_means_2d)



In [ ]:
for topic, genes in genes_by_topic.items():
    print(f"Scoring topic {topic} with {len(genes)} genes")
    genes_symbol = combined_adata.var_names[combined_adata.var_names.isin(genes)].tolist()
    print(f"Found {len(genes_symbol)} genes in adata.var_names")
    sc.tl.score_genes(combined_adata, genes_symbol, score_name=f"Signature_{topic}", use_raw=False)

In [ ]:
for topic, genes in exclusive_genes_by_topic.items():
    print(f"Scoring topic {topic} with {len(genes)} exclusive genes")
    genes_symbol = combined_adata.var_names[combined_adata.var_names.isin(genes)].tolist()
    print(f"Found {len(genes_symbol)} genes in adata.var_names")
    sc.tl.score_genes(combined_adata, genes_symbol, score_name=f"SignatureUnique_{topic}", use_raw=False)
    print(f"Scored topic {topic} with {len(genes_symbol)} exclusive genes")

In [ ]:
import os

# Extract all columns from sub_adata.obs that start with "Signature"
signature_cols = [col for col in combined_adata.obs.columns if col.startswith("Signature")]
signature_df = combined_adata.obs[signature_cols]

# Save as pickle, equivalent to RDS in R
signature_pickle_path = os.path.join(plots_dir, "Signature.pickle")
signature_df.to_pickle(signature_pickle_path)
signature_df.to_csv(os.path.join(plots_dir, "Signature.csv"))

In [ ]:
import json

pathway_file_format = os.path.join(get_result_dir(), 'topic_space_plots', 'Pathways_genes_{cell_type}.json')

if cell_type == 'excitatory':
    with open(pathway_file_format.format(cell_type='cux2p')) as f:
        pathways = json.load(f)

    with open(pathway_file_format.format(cell_type='cux2m')) as f:
        new_pathways = json.load(f)
        for k, v in new_pathways.items():
            if k in pathways:
                # Union of gene lists
                pathways[k] = list(set(pathways[k]).union(set(v)))
            else:
                pathways[k] = v


else:
    with open(pathway_file_format.format(cell_type=cell_type)) as f:
        pathways = json.load(f)


if cell_type in ["inhibitory", "excitatory"]:
    with open(f'/ems/elsc-labs/habib-n/roi.meir/500/Plots//natasha/cuimc2/{cell_type}/PathwayToGenes.json') as f:
        neuornal_pathways = json.load(f)

    

print(pathways)

In [ ]:
for k in pathways.keys():
    if k not in neuornal_pathways:
        print(k)
    else:
        assert set(neuornal_pathways[k]) == set(pathways[k]), f"neuornal_pathways and pathways have different values ({k})"

pathways.update(neuornal_pathways)

In [ ]:

for pathway_name, genes in pathways.items():
    if not isinstance(genes, list):
        genes = [genes]
    if f"Pathway_{pathway_name}" in combined_adata.obs.columns:
        print(f"Pathway {pathway_name} already scored")
        continue

    print(f"Scoring pathway {pathway_name} with {len(genes)} genes")
    genes_symbol = combined_adata.var_names[combined_adata.var_names.isin(genes)].tolist()

    print(f"Found {len(genes_symbol)}/{len(genes)} genes in adata.var_names")
    sc.tl.score_genes(combined_adata, genes_symbol, score_name=f"Pathway_{pathway_name}", use_raw=False)
    print(f"Scored pathway {pathway_name} with {len(genes_symbol)} genes")

In [ ]:
pathway_cols = [col for col in combined_adata.obs.columns if col.startswith("Pathway")]
pathway_df = combined_adata.obs[pathway_cols]
print(f"Number of pathways: {len(pathway_df.columns)}")

pathway_pickle_path = os.path.join(plots_dir, "Pathway.pickle")
pathway_df.to_pickle(pathway_pickle_path)
pathway_df.to_csv(os.path.join(plots_dir, "Pathway.csv"))

In [ ]:
signature_df = pd.read_pickle(os.path.join(plots_dir, "Signature.pickle"))
print(signature_df.columns)

if not signature_df.columns.isin(combined_adata.obs.columns).all():
    combined_adata.obs = combined_adata.obs.join(signature_df)

pathway_df = pd.read_pickle(os.path.join(plots_dir, "Pathway.pickle"))
print(pathway_df.columns)
if not pathway_df.columns.isin(combined_adata.obs.columns).all():
    combined_adata.obs = combined_adata.obs.join(pathway_df)


In [ ]:
# Save pathways and signatures together with subject, cell id and cell type
cols = [col for col in combined_adata.obs.columns if col.startswith("Pathway") or col.startswith("Signature") or col == "subject" or col == "cell_type_high_resolution"]

print(cols)

df_to_save = combined_adata.obs[cols]
df_to_save.to_csv(os.path.join(plots_dir, "pathways_and_signatures.csv"))
